In [ ]:
!pip install facenet_pytorch

In [ ]:
!pip install --upgrade Pillow
!pip install torchvision torchaudio --no-deps
!pip install "thinc<8.3.6"
!pip install --upgrade facenet_pytorch

  Using cached pillow-11.2.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (8.9 kB)
Using cached pillow-11.2.1-cp311-cp311-manylinux_2_28_x86_64.whl (4.6 MB)
  Attempting uninstall: Pillow
    Found existing installation: pillow 10.2.0
    Uninstalling pillow-10.2.0:
      Successfully uninstalled pillow-10.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
facenet-pytorch 2.6.0 requires Pillow<10.3.0,>=10.2.0, but you have pillow 11.2.1 which is incompatible.


  Using cached pillow-10.2.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (9.7 kB)
Using cached pillow-10.2.0-cp311-cp311-manylinux_2_28_x86_64.whl (4.5 MB)
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.2.1
    Uninstalling pillow-11.2.1:
      Successfully uninstalled pillow-11.2.1


In [ ]:
from google.colab import files
import cv2
import numpy as np
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image
import os
import time
import shutil

# ====== 2. UPLOAD TARGET IMAGE AND VIDEO ======
print("⬆ Upload the TARGET FACE image (person to search for):")
target_image = files.upload()
target_name = list(target_image.keys())[0]

print("\n⬆ Upload the VIDEO to search through:")
video_file = files.upload()
video_name = list(video_file.keys())[0]

start_time = time.time()

# ====== 3. SETUP MTCNN AND FACENET ======
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)
mtcnn = MTCNN(keep_all=True, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

# ====== 4. GET TARGET FACE EMBEDDING (FIXED) ======
target_img = Image.open(target_name).convert('RGB')
all_faces = mtcnn(target_img)

if all_faces is None:
    raise ValueError("No face detected in target image!")

# pick the first detected face
if all_faces.ndimension() == 4:
    face_tensor = all_faces[0]
else:
    face_tensor = all_faces

# now make it a batch of 1
face_tensor = face_tensor.unsqueeze(0).to(device)
target_embedding = resnet(face_tensor).detach().cpu()

# ====== 5. PROCESS VIDEO FRAME-BY-FRAME ======
cap = cv2.VideoCapture(video_name)
fps = cap.get(cv2.CAP_PROP_FPS)
skip_frames = int(fps / 7)

threshold = 0.7
frame_count = 0
matched_video_writer = None
saved_frames_dir = "matched_frames"
os.makedirs(saved_frames_dir, exist_ok=True)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
output_filename = "matched_output_video.mp4"

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    if frame_count % skip_frames != 0:
        continue

    # convert and detect
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    boxes, _ = mtcnn.detect(pil)
    faces = mtcnn(pil)

    if boxes is not None and faces is not None:
        # faces: [N,3,160,160] tensor
        embeddings = resnet(faces.to(device)).detach().cpu()  # [N,512]
        for i, box in enumerate(boxes):
            similarity = torch.nn.functional.cosine_similarity(
                target_embedding,
                embeddings[i].unsqueeze(0),
            ).item()

            if similarity > threshold:
                x1, y1, x2, y2 = map(int, box)

                # draw rectangle
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                # draw score
                cv2.putText(frame, f"{similarity:.2f}", (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

                print(f" Match at frame {frame_count}, sim={similarity:.2f}")

                # save frame image
                timestamp = round(frame_count / fps, 2)
                img_path = os.path.join(
                    saved_frames_dir,
                    f"matched_{frame_count}_t{timestamp}.jpg"
                )
                cv2.imwrite(img_path, frame)

                # init writer if needed
                if matched_video_writer is None:
                    h, w = frame.shape[:2]
                    matched_video_writer = cv2.VideoWriter(
                        output_filename, fourcc, fps, (w, h)
                    )
                matched_video_writer.write(frame)

                break  # only one box per frame

cap.release()
if matched_video_writer:
    matched_video_writer.release()

# ====== 6. DOWNLOAD RESULTS ======
if os.path.exists(output_filename):
    print("\n Downloading matched video…")
    files.download(output_filename)
else:
    print("\n No matches found in video.")

if os.path.isdir(saved_frames_dir):
    print(" Downloading matched frames zip…")
    shutil.make_archive(saved_frames_dir, 'zip', saved_frames_dir)
    files.download(f"{saved_frames_dir}.zip")

print(f"\n Done in {time.time() - start_time:.2f}s")

⬆ Upload the TARGET FACE image (person to search for):


Saving target_name.jpg to target_name.jpg

⬆ Upload the VIDEO to search through:


Saving video_name.mp4 to video_name.mp4
Using device: cuda


  0%|          | 0.00/107M [00:00<?, ?B/s]

✅ Match at frame 243, sim=0.75
✅ Match at frame 246, sim=0.71
✅ Match at frame 249, sim=0.74
✅ Match at frame 252, sim=0.74
✅ Match at frame 255, sim=0.82
✅ Match at frame 258, sim=0.94
✅ Match at frame 261, sim=0.85
✅ Match at frame 264, sim=0.80
✅ Match at frame 267, sim=0.78
✅ Match at frame 270, sim=0.91
✅ Match at frame 273, sim=0.85
✅ Match at frame 276, sim=0.78
✅ Match at frame 279, sim=0.71
✅ Match at frame 282, sim=0.78

✅ Downloading matched video…


...

<IPython.core.display.Javascript object>

✅ Downloading matched frames zip…


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Done in 44.65s
